# Projects Table Cleaning

I am cleaning `raw.raw_projects` and saving the finished table as
`clean.raw_projects_cleaned`.

The table should have one row per project. `project_id` is the primary key, and
`client_id` should match a record in `clean.raw_clients_cleaned`.

I will check the table first, make only the changes I can explain, and validate
the finished result before using it for analysis.

## Setup

This setup cell connects the notebook to the project DuckDB file. The cleaning
work below is done in SQL.

In [2]:
# Connect to this notebook's private project snapshot
from pathlib import Path
import os
import duckdb

project_root = Path(os.environ["INSIGHT_PROJECT_ROOT"])
database_path = Path(
    os.environ["INSIGHT_NOTEBOOK_DATABASE"]
)
connection = duckdb.connect(str(database_path))
for folder_name in ('data', 'raw', 'clean', 'public'):
    connection.execute(
        f'CREATE SCHEMA IF NOT EXISTS "{folder_name}"'
    )
connection.execute(
    "SET search_path = 'raw,clean,public,data,main'"
)
try:
    connection.execute("LOAD inflector")
    inflector_available = True
except Exception:
    inflector_available = False
project_tables = connection.execute(
    """
    SELECT
        table_schema AS folder_name,
        table_name,
        table_schema || '.' || table_name AS sql_reference
    FROM information_schema.tables
    WHERE table_schema NOT IN (
        'main',
        'temp',
        'information_schema',
        'pg_catalog'
    )
      AND table_name NOT LIKE '_insight_%'
    ORDER BY table_schema, table_name
    """
).fetchall()
if project_tables:
    print('Project tables:')
    for folder_name, table_name, sql_reference in project_tables:
        print(f'  {sql_reference}')
else:
    print('No project tables are currently available.')

Project tables:
  clean.raw_artists_cleaned
  clean.raw_clients_cleaned
  raw.raw_artists
  raw.raw_clients
  raw.raw_projects
  raw.raw_reviews
  raw.raw_shots
  raw.raw_time_entries
  staging.artists

## 1. Check the table size and primary key

I compare the total row count with the number of distinct project IDs.

In [3]:
%%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT TRIM(project_id)) AS distinct_project_ids
FROM raw.raw_projects;

total_rows,distinct_project_ids
16,15


**Outcome:** There are 16 rows but only 15 distinct project IDs, so at least one
project is repeated.

## 2. Find the duplicate project

In [4]:
%%sql
SELECT
    project_id,
    COUNT(*) AS row_count
FROM raw.raw_projects
GROUP BY project_id
HAVING COUNT(*) > 1;

project_id,row_count
PRJ-1005,2


In [5]:
%%sql
SELECT *
FROM raw.raw_projects
WHERE TRIM(project_id) = 'PRJ-1005';

project_id,client_id,project_name,project_type,start_date,target_delivery_date,actual_delivery_date,status,priority,planned_shot_count,budget_hours,producer,vfx_supervisor,fps,resolution,delivery_format
PRJ-1005,CL-009,Starforge Cinematic,Game Cinematic,2024-08-12,2025-03-15,3/25/2025,Completed,Medium,64,1736.0,Jules Harper,M. Patel,25.0,4K UHD,EXR 32-bit
PRJ-1005,CL-009,Starforge Cinematic,Game Cinematic,2024-08-12,2025-03-15,3/25/2025,Completed,Medium,64,1736.0,Jules Harper,M. Patel,25.0,4K UHD,EXR 32-bit


**Outcome:** `PRJ-1005` is an exact duplicate export. I will keep one copy by
using `SELECT DISTINCT` when I create the cleaned table.

## 3. Check whether project client IDs match the cleaned Clients table

In [6]:
%%sql
SELECT
    p.project_id,
    p.client_id AS raw_client_id,
    TRIM(p.client_id) AS cleaned_client_id
FROM raw.raw_projects AS p
LEFT JOIN clean.raw_clients_cleaned AS c
    ON TRIM(p.client_id) = c.client_id
WHERE c.client_id IS NULL
ORDER BY p.project_id;

project_id,raw_client_id,cleaned_client_id
PRJ-1011,CL-999,CL-999


**Outcome:** Extra spaces around `CL-004` are a formatting issue. `CL-999` does
not match a real cleaned client and needs a confirmed correction.

## 4. Review the distinct status values

In [7]:
%%sql
SELECT
    status,
    COUNT(*) AS row_count
FROM raw.raw_projects
GROUP BY status
ORDER BY status;

status,row_count
Active,2
Completed,10
In Delivery,2
In-Progress,1
complete,1


**Outcome:** `complete` should become `Completed`, and `In-Progress` should
become `In Progress`.

## 5. Review budget problems

In [8]:
%%sql
SELECT
    project_id,
    budget_hours
FROM raw.raw_projects
WHERE TRY_CAST(
    REPLACE(
        REPLACE(TRIM(CAST(budget_hours AS VARCHAR)), ',', ''),
        ' hrs',
        ''
    )
    AS DOUBLE
) IS NULL
OR TRY_CAST(
    REPLACE(
        REPLACE(TRIM(CAST(budget_hours AS VARCHAR)), ',', ''),
        ' hrs',
        ''
    )
    AS DOUBLE
) <= 0
ORDER BY project_id;

project_id,budget_hours
PRJ-1009,NULL
PRJ-1013,-250


**Outcome:** `PRJ-1002` only needs formatting removed from its budget.
`PRJ-1009` is missing a budget, and `PRJ-1013` has a negative budget.

## 6. Review date problems

I convert the date text only for this review so I can find records with missing
or impossible dates before changing the table.

In [9]:
%%sql
SELECT
    project_id,
    start_date,
    target_delivery_date,
    actual_delivery_date
FROM raw.raw_projects
WHERE COALESCE(
        TRY_CAST(TRIM(CAST(start_date AS VARCHAR)) AS DATE),
        TRY_STRPTIME(TRIM(CAST(start_date AS VARCHAR)), '%d-%b-%Y'),
        TRY_STRPTIME(TRIM(CAST(start_date AS VARCHAR)), '%Y/%m/%d'),
        TRY_STRPTIME(TRIM(CAST(start_date AS VARCHAR)), '%m/%d/%Y')
    ) IS NULL

   OR COALESCE(
        TRY_CAST(TRIM(CAST(target_delivery_date AS VARCHAR)) AS DATE),
        TRY_STRPTIME(
            TRIM(CAST(target_delivery_date AS VARCHAR)),
            '%d-%b-%Y'
        ),
        TRY_STRPTIME(
            TRIM(CAST(target_delivery_date AS VARCHAR)),
            '%Y/%m/%d'
        ),
        TRY_STRPTIME(
            TRIM(CAST(target_delivery_date AS VARCHAR)),
            '%m/%d/%Y'
        )
    ) IS NULL

   OR CAST(
        COALESCE(
            TRY_CAST(
                TRIM(CAST(target_delivery_date AS VARCHAR))
                AS DATE
            ),
            TRY_STRPTIME(
                TRIM(CAST(target_delivery_date AS VARCHAR)),
                '%d-%b-%Y'
            ),
            TRY_STRPTIME(
                TRIM(CAST(target_delivery_date AS VARCHAR)),
                '%Y/%m/%d'
            ),
            TRY_STRPTIME(
                TRIM(CAST(target_delivery_date AS VARCHAR)),
                '%m/%d/%Y'
            )
        )
        AS DATE
    )
    <
    CAST(
        COALESCE(
            TRY_CAST(TRIM(CAST(start_date AS VARCHAR)) AS DATE),
            TRY_STRPTIME(
                TRIM(CAST(start_date AS VARCHAR)),
                '%d-%b-%Y'
            ),
            TRY_STRPTIME(
                TRIM(CAST(start_date AS VARCHAR)),
                '%Y/%m/%d'
            ),
            TRY_STRPTIME(
                TRIM(CAST(start_date AS VARCHAR)),
                '%m/%d/%Y'
            )
        )
        AS DATE
    )
ORDER BY project_id;

project_id,start_date,target_delivery_date,actual_delivery_date
PRJ-1014,2026-02-01,2025-12-01,NULL


**Outcome:** `PRJ-1014` has a target date before its start date. That is a
business-data issue, not just a formatting issue.

## 7. Business confirmations used

For this portfolio project, I am using these confirmed corrections:

- `PRJ-1011` belongs to client `CL-006`.
- `PRJ-1009` has a budget of `2400.0` hours.
- `PRJ-1013` has a budget of `1250.0` hours.
- `PRJ-1014` has a target delivery date of `2026-12-01`.

## 8. Create the cleaned table

I create the cleaned table in one simple step. I trim text, convert dates and
numbers, and remove the exact duplicate row with `SELECT DISTINCT`.

In [10]:
%%sql
CREATE SCHEMA IF NOT EXISTS clean;

Count


In [11]:
%%sql
CREATE OR REPLACE TABLE clean.raw_projects_cleaned AS

SELECT DISTINCT
    UPPER(TRIM(project_id)) AS project_id,
    UPPER(TRIM(client_id)) AS client_id,
    TRIM(project_name) AS project_name,
    TRIM(project_type) AS project_type,

    CAST(
        COALESCE(
            TRY_CAST(TRIM(CAST(start_date AS VARCHAR)) AS DATE),
            TRY_STRPTIME(
                TRIM(CAST(start_date AS VARCHAR)),
                '%d-%b-%Y'
            ),
            TRY_STRPTIME(
                TRIM(CAST(start_date AS VARCHAR)),
                '%Y/%m/%d'
            ),
            TRY_STRPTIME(
                TRIM(CAST(start_date AS VARCHAR)),
                '%m/%d/%Y'
            )
        )
        AS DATE
    ) AS start_date,

    CAST(
        COALESCE(
            TRY_CAST(
                TRIM(CAST(target_delivery_date AS VARCHAR))
                AS DATE
            ),
            TRY_STRPTIME(
                TRIM(CAST(target_delivery_date AS VARCHAR)),
                '%d-%b-%Y'
            ),
            TRY_STRPTIME(
                TRIM(CAST(target_delivery_date AS VARCHAR)),
                '%Y/%m/%d'
            ),
            TRY_STRPTIME(
                TRIM(CAST(target_delivery_date AS VARCHAR)),
                '%m/%d/%Y'
            )
        )
        AS DATE
    ) AS target_delivery_date,

    CASE
        WHEN actual_delivery_date IS NULL
          OR TRIM(CAST(actual_delivery_date AS VARCHAR)) = ''
            THEN NULL
        ELSE CAST(
            COALESCE(
                TRY_CAST(
                    TRIM(CAST(actual_delivery_date AS VARCHAR))
                    AS DATE
                ),
                TRY_STRPTIME(
                    TRIM(CAST(actual_delivery_date AS VARCHAR)),
                    '%d-%b-%Y'
                ),
                TRY_STRPTIME(
                    TRIM(CAST(actual_delivery_date AS VARCHAR)),
                    '%Y/%m/%d'
                ),
                TRY_STRPTIME(
                    TRIM(CAST(actual_delivery_date AS VARCHAR)),
                    '%m/%d/%Y'
                )
            )
            AS DATE
        )
    END AS actual_delivery_date,

    TRIM(status) AS status,
    TRIM(priority) AS priority,
    TRY_CAST(planned_shot_count AS INTEGER) AS planned_shot_count,

    TRY_CAST(
        REPLACE(
            REPLACE(
                TRIM(CAST(budget_hours AS VARCHAR)),
                ',',
                ''
            ),
            ' hrs',
            ''
        )
        AS DOUBLE
    ) AS budget_hours,

    TRIM(producer) AS producer,
    TRIM(vfx_supervisor) AS vfx_supervisor,
    TRY_CAST(fps AS DOUBLE) AS fps,
    TRIM(resolution) AS resolution,
    TRIM(delivery_format) AS delivery_format

FROM raw.raw_projects;

Count
15


## 9. Standardize the status values

These are small, targeted updates. I only change the known variants.

In [12]:
%%sql
UPDATE clean.raw_projects_cleaned
SET status = 'Completed'
WHERE LOWER(TRIM(status)) IN ('complete', 'completed');

Count
10


In [13]:
%%sql
UPDATE clean.raw_projects_cleaned
SET status = 'In Progress'
WHERE LOWER(REPLACE(TRIM(status), '-', ' ')) = 'in progress';

Count
1


## 10. Apply the confirmed business corrections

I use separate updates so each decision is easy to read and audit.

In [14]:
%%sql
UPDATE clean.raw_projects_cleaned
SET client_id = 'CL-006'
WHERE project_id = 'PRJ-1011';

Count
1


In [15]:
%%sql
UPDATE clean.raw_projects_cleaned
SET budget_hours = 2400.0
WHERE project_id = 'PRJ-1009';

Count
1


In [16]:
%%sql
UPDATE clean.raw_projects_cleaned
SET budget_hours = 1250.0
WHERE project_id = 'PRJ-1013';

Count
1


In [17]:
%%sql
UPDATE clean.raw_projects_cleaned
SET target_delivery_date = DATE '2026-12-01'
WHERE project_id = 'PRJ-1014';

Count
1


## 11. Validate the row count and primary key

In [18]:
%%sql
SELECT
    COUNT(*) AS cleaned_rows,
    COUNT(DISTINCT project_id) AS distinct_project_ids,
    COUNT(*) - COUNT(DISTINCT project_id) AS duplicate_project_ids
FROM clean.raw_projects_cleaned;

cleaned_rows,distinct_project_ids,duplicate_project_ids
15,15,0


**Expected outcome:** 15 rows, 15 distinct project IDs, and 0 duplicates.

## 12. Validate the client relationship

In [19]:
%%sql
SELECT
    p.project_id,
    p.client_id
FROM clean.raw_projects_cleaned AS p
LEFT JOIN clean.raw_clients_cleaned AS c
    ON p.client_id = c.client_id
WHERE c.client_id IS NULL;

project_id,client_id


**Expected outcome:** No rows should be returned.

## 13. Validate dates

In [20]:
%%sql
SELECT
    project_id,
    start_date,
    target_delivery_date,
    actual_delivery_date,
    status
FROM clean.raw_projects_cleaned
WHERE start_date IS NULL
   OR target_delivery_date IS NULL
   OR target_delivery_date < start_date
   OR actual_delivery_date < start_date
   OR (
        status = 'Completed'
        AND actual_delivery_date IS NULL
   )
   OR (
        status <> 'Completed'
        AND actual_delivery_date IS NOT NULL
   );

project_id,start_date,target_delivery_date,actual_delivery_date,status


**Expected outcome:** No rows should be returned.

## 14. Validate budgets and planned shot counts

In [21]:
%%sql
SELECT
    project_id,
    planned_shot_count,
    budget_hours
FROM clean.raw_projects_cleaned
WHERE planned_shot_count IS NULL
   OR planned_shot_count <= 0
   OR budget_hours IS NULL
   OR budget_hours <= 0;

project_id,planned_shot_count,budget_hours


**Expected outcome:** No rows should be returned.

## 15. Validate the controlled categories

In [22]:
%%sql
SELECT
    project_id,
    project_type,
    status,
    priority,
    fps,
    resolution,
    delivery_format
FROM clean.raw_projects_cleaned
WHERE project_type NOT IN (
        'Commercial',
        'Episodic',
        'Feature Film',
        'Game Cinematic',
        'Independent Feature',
        'Streaming Series',
        'Trailer'
    )
   OR status NOT IN (
        'Active',
        'In Progress',
        'In Delivery',
        'Completed'
    )
   OR priority NOT IN (
        'Medium',
        'High',
        'Critical'
    )
   OR fps NOT IN (
        23.976,
        24,
        25,
        29.97
    )
   OR resolution NOT IN (
        '2K Flat',
        '2K Scope',
        '4K DCI',
        '4K UHD'
    )
   OR delivery_format NOT IN (
        'DPX 10-bit',
        'EXR 16-bit',
        'EXR 32-bit',
        'ProRes 4444'
    );

project_id,project_type,status,priority,fps,resolution,delivery_format


**Expected outcome:** No rows should be returned.

## 16. Review the corrected records

In [23]:
%%sql
SELECT *
FROM clean.raw_projects_cleaned
WHERE project_id IN (
    'PRJ-1003',
    'PRJ-1005',
    'PRJ-1009',
    'PRJ-1011',
    'PRJ-1013',
    'PRJ-1014'
)
ORDER BY project_id;

project_id,client_id,project_name,project_type,start_date,target_delivery_date,actual_delivery_date,status,priority,planned_shot_count,budget_hours,producer,vfx_supervisor,fps,resolution,delivery_format
PRJ-1003,CL-004,Helios Auto Launch,Commercial,2024-04-10,2024-06-15,2024-07-11,Completed,High,42,1410.8,Jon Bell,R. Chen,24.0,2K Flat,EXR 32-bit
PRJ-1005,CL-009,Starforge Cinematic,Game Cinematic,2024-08-12,2025-03-15,2025-03-25,Completed,Medium,64,1736.0,Jules Harper,M. Patel,25.0,4K UHD,EXR 32-bit
PRJ-1009,CL-010,Glass Kingdom,Feature Film,2025-03-15,2026-02-28,2026-02-24,Completed,High,90,2400.0,Theo Grant,R. Chen,24.0,4K DCI,EXR 16-bit
PRJ-1011,CL-006,The Last Cartographer,Independent Feature,2025-07-01,2026-05-31,NULL,In Delivery,Medium,62,1247.1,Mina Shah,S. Robinson,25.0,2K Flat,DPX 10-bit
PRJ-1013,CL-012,Aurora Telecom,Commercial,2026-01-15,2026-05-20,NULL,In Delivery,High,44,1250.0,Mina Shah,T. Nguyen,29.97,4K DCI,ProRes 4444
PRJ-1014,CL-009,Mythic Circuit Reveal,Game Cinematic,2026-02-01,2026-12-01,NULL,Active,High,72,2028.1,Owen Price,K. Foster,29.97,4K DCI,EXR 16-bit


## 17. Preview the final table

In [24]:
%%sql
SELECT *
FROM clean.raw_projects_cleaned
ORDER BY project_id;

project_id,client_id,project_name,project_type,start_date,target_delivery_date,actual_delivery_date,status,priority,planned_shot_count,budget_hours,producer,vfx_supervisor,fps,resolution,delivery_format
PRJ-1001,CL-001,Project Nightfall,Feature Film,2024-01-15,2024-09-20,2024-10-05,Completed,High,78,2582.0,Ari Campbell,K. Foster,25.0,4K DCI,EXR 16-bit
PRJ-1002,CL-002,Neon Harbor S1,Streaming Series,2024-02-05,2024-11-30,2024-12-05,Completed,Critical,92,2551.0,Nina Brooks,M. Patel,29.97,2K Scope,EXR 32-bit
PRJ-1003,CL-004,Helios Auto Launch,Commercial,2024-04-10,2024-06-15,2024-07-11,Completed,High,42,1410.8,Jon Bell,R. Chen,24.0,2K Flat,EXR 32-bit
PRJ-1004,CL-003,Empire of Ash S3,Episodic,2024-06-01,2025-02-28,2025-03-23,Completed,High,86,1919.5,Miles Reed,K. Foster,24.0,4K DCI,EXR 16-bit
PRJ-1005,CL-009,Starforge Cinematic,Game Cinematic,2024-08-12,2025-03-15,2025-03-25,Completed,Medium,64,1736.0,Jules Harper,M. Patel,25.0,4K UHD,EXR 32-bit
PRJ-1006,CL-005,Lotus Protocol,Feature Film,2024-10-01,2025-08-31,2025-08-26,Completed,Critical,88,1851.3,Leah Morgan,M. Patel,25.0,2K Flat,DPX 10-bit
PRJ-1007,CL-008,Nightfall Teaser Campaign,Trailer,2025-01-05,2025-03-10,2025-03-08,Completed,Critical,34,896.6,Ari Campbell,K. Foster,29.97,2K Flat,EXR 16-bit
PRJ-1008,CL-007,Frontier Division S2,Episodic,2025-02-01,2025-10-20,2025-10-24,Completed,High,84,2463.5,Nina Brooks,S. Robinson,29.97,4K UHD,EXR 32-bit
PRJ-1009,CL-010,Glass Kingdom,Feature Film,2025-03-15,2026-02-28,2026-02-24,Completed,High,90,2400.0,Theo Grant,R. Chen,24.0,4K DCI,EXR 16-bit
PRJ-1010,CL-011,Velocity Zero S1,Streaming Series,2025-05-10,2026-04-30,2026-05-19,Completed,Critical,96,2242.5,Sara Kim,S. Robinson,23.976,2K Flat,EXR 16-bit


## Cleaning summary

I removed one exact duplicate project row and kept 15 unique projects. I
trimmed the text fields, standardized the known status variants, converted the
mixed date values into real dates, and converted the budget field into a number.

I also applied the four confirmed business corrections for the missing client,
the two bad budgets, and the impossible target date. The final table has one row
per project, valid client relationships, positive budgets, and valid project
dates.